In [2]:
%pip install tensorflow

   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 2.9/351.2 MB 22.9 MB/s eta 0:00:16
    --------------------------------------- 5.0/351.2 MB 13.9 MB/s eta 0:00:25
    --------------------------------------- 6.3/351.2 MB 10.9 MB/s eta 0:00:32
    --------------------------------------- 7.3/351.2 MB 9.5 MB/s eta 0:00:37
    --------------------------------------- 8.4/351.2 MB 8.7 MB/s eta 0:00:40
   - -------------------------------------- 9.7/351.2 MB 8.2 MB/s eta 0:00:42
   - -------------------------------------- 11.0/351.2 MB 7.8 MB/s eta 0:00:44
   - -------------------------------------- 11.8/351.2 MB 7.6 MB/s eta 0:00:45
   - -------------------------------------- 11.8/351.2 MB 7.6 MB/s eta 0:00:45
   - -------------------------------------- 11.8/351.2 MB 7.6 MB/s eta 0:00:45
   - -------------------------------------- 11.8/351.2 MB 7.6 MB/s eta 0:00:45
   - -------------------------------------- 11.8/351.2 MB 7.6 M

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import binascii
import os

# --- Parameters ---
IMG_SIZE = 96
BATCH_SIZE = 32
DATA_DIR = "./dataset-resized" # Path to your TrashNet dataset

# cardboard, glass, metal, paper, plastic, trash
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    color_mode="grayscale", # Grayscale to save memory
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="training",
    seed=123
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    color_mode="grayscale",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=123
)

# Get class names for reference
class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# Normalize
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))



# --- 2. Build Model ---
model = models.Sequential([
    layers.InputLayer(input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Conv2D(8, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(16, activation='relu'),
    layers.Dense(num_classes, activation='softmax') # Multi-class output
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# --- 3. Train Model ---
print("Training model...")
model.fit(train_ds, validation_data=val_ds, epochs=15)

# --- 4. Quantize to INT8 ---
print("Quantizing model...")
def representative_dataset_gen():
    for images, labels in train_ds.take(10):
        yield [images]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

# --- 5. Generate C Header ---
def convert_to_c_array(bytes_data, array_name="g_model"):
    hex_data = binascii.hexlify(bytes_data).decode('utf-8')
    c_array = f"const unsigned char {array_name}[] = {{\n"
    for i in range(0, len(hex_data), 2):
        c_array += f"0x{hex_data[i:i+2]}, "
        if (i + 2) % 24 == 0:
            c_array += "\n"
    c_array += f"}};\nconst int {array_name}_len = {len(bytes_data)};\n"
    with open(f"{array_name}.h", "w") as f:
        f.write(c_array)

convert_to_c_array(tflite_quant_model)
print("Model saved as g_model.h")

Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Training model...
Epoch 1/15


C:\Users\pramo\anaconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
C:\Users\pramo\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


64/64 ━━━━━━━━━━━━━━━━━━━━ 14s 169ms/step - accuracy: 0.1899 - loss: 1.7704 - val_accuracy: 0.1743 - val_loss: 1.7290
Epoch 2/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.2428 - loss: 1.7014 - val_accuracy: 0.2594 - val_loss: 1.6697
Epoch 3/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.3046 - loss: 1.6125 - val_accuracy: 0.3564 - val_loss: 1.5988
Epoch 4/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.3635 - loss: 1.5160 - val_accuracy: 0.3465 - val_loss: 1.5117
Epoch 5/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - accuracy: 0.3887 - loss: 1.4695 - val_accuracy: 0.4099 - val_loss: 1.4449
Epoch 6/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - accuracy: 0.4105 - loss: 1.4108 - val_accuracy: 0.4257 - val_loss: 1.4162
Epoch 7/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - accuracy: 0.4313 - loss: 1.3862 - val_accuracy: 0.4139 - val_loss: 1.4341
Epoch 8/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - accuracy: 0.4466 - loss: 1.3452 - val_accuracy: 0.4337 - val_loss: 

INFO:tensorflow:Assets written to: C:\Users\pramo\AppData\Local\Temp\tmpkn6sj7mi\assets


Saved artifact at 'C:\Users\pramo\AppData\Local\Temp\tmpkn6sj7mi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2351813851472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2351813851664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2351813851088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2351813851280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2351813851856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2351813852432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2351815442896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2351815443088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2351815455376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2351815448848: TensorSpec(shape=(), dtype=tf.resource, name=None)


C:\Users\pramo\anaconda3\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Model saved as g_model.h
